# AI Document Assistant (RAG Pipeline)

**Objective:** Build a Retrieval-Augmented Generation (RAG) system that allows users to "chat" with complex, proprietary PDF documents (such as legal contracts, corporate manuals, or financial reports).

**Architecture:** 1. **Document Ingestion:** Utilizes `PyPDF` to parse the document and `LangChain` to split the text into optimized, readable chunks.
2. **Vector Database:** Converts text chunks into high-dimensional embeddings using Hugging Face's `all-MiniLM-L6-v2` model, storing them in a local `FAISS` vector database for lightning-fast semantic search.
3. **LLM Integration:** Connects the vector database to Google's Gemini Large Language Model. When a user asks a question, the system retrieves only the relevant paragraphs from the FAISS database and feeds them to Gemini to generate a highly accurate, hallucination-free answer strictly based on the provided document.

In [30]:
import os
from dotenv import load_dotenv

load_dotenv()

my_api_key = os.environ.get("GOOGLE_API_KEY")

print("Secrets loaded safely!")

Secrets loaded safely!


In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic.chains import RetrievalQA

pdf_name = "test_document.pdf"
print(f"Loading {pdf_name}...")
loader = PyPDFLoader(pdf_name)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
print(f"Split document into {len(texts)} chunks.")

print("Building the FAISS vector database...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(texts, embeddings)

print("Connecting Gemini LLM...")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 3})
)

print("✅ RAG Pipeline is active and ready for queries!")

Loading test_document.pdf...
Split document into 3217 chunks.
Building the FAISS vector database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1093.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting Gemini LLM...
✅ RAG Pipeline is active and ready for queries!


In [ ]:
question = "What is the primary subject of this document? Please summarize the core points in 3 bullet points."

print(f"User Query: {question}\n")
print("-" * 50)

# Run the query through the RAG pipeline
response = qa_chain.invoke(question)

print("AI Response:")
print(response['result'])

User Query: What is the primary subject of this document? Please summarize the core points in 3 bullet points.

--------------------------------------------------
AI Response:
The primary subject of this document is **mathematics, specifically focusing on functions and their graphical representation, particularly with the aid of software.**

Here are the core points:

*   The document is an excerpt from "Chapter 1 Functions" of an educational textbook, as indicated by the chapter title and ISBN.
*   It includes a section titled "Section 1.4 Graphing with Software," highlighting the use of technology for visualizing functions.
*   It provides an example of transforming an equation (y - x^2 = 16) into functions suitable for graphing (y = ±sqrt(x^2 + 16)) and specifies how to graph a particular branch (the upper branch).
